In [13]:
train_data = {'d e e p </w>': 5, 
'l e a r n i n g </w>': 7, 
'n a t u r a l </w>': 6, 
'l a n g u a g e </w>': 3,
'p r o c e s s i n g </w>':7}

In [14]:
bpe_vocab = {'a', 
'e', 
'p', 
'</w>', 
'g', 'o', 's', 'l', 'r', 'u', 'd', 'n', 'i', 't', 'c'}

In [15]:
train_data = {'d e e p </w>': 5, 'l e a r n i ng </w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3,'p r o c e s s i ng </w>':7}
bpe_vocab = {'a', 'e', 'p', '</w>', 'g', 'o', 's', 'l', 'r', 'u', 'd', 'n', 'i', 't', 'c', 'ng'}

In [16]:
train_data = {'d e e p </w>': 5, 'l e a r n ing </w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3,'p r o c e s s ing </w>':7}
bpe_vocab = {'a', 'e', 'p', '</w>', 'g', 'o', 's', 'l', 'r', 'u', 'd', 'n', 't', 'c', 'ng', 'ing'}

In [17]:
train_data = {'d e e p </w>': 5, 'l e a r n ing</w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3,'p r o c e s s ing</w>':7}
bpe_vocab = {'a', 'e', 'p', '</w>', 'g', 'o', 's', 'l', 'r', 'u', 'd', 'n', 't', 'c', 'ng','ing</w>'}

In [18]:
import re
import collections

def get_subwords(data:dict):
    subwards = collections.defaultdict(int)
    for key,values in data.items():
        for word in key.split():
            subwards[word]+=values
        
    return subwards
train_data = {'d e e p </w>': 5, 'l e a r n i n g </w>': 7, 'n a t u r a l </w>': 6, 'l a n g u a g e </w>': 3,'p r o c e s s i n g </w>':7}
subwords = get_subwords(train_data)

bpe_vocab = set(subwords.keys())
bpe_vocab

{'</w>', 'a', 'c', 'd', 'e', 'g', 'i', 'l', 'n', 'o', 'p', 'r', 's', 't', 'u'}

In [19]:
def get_pair_with_frequency(data:dict[str,int]):
    pairwords = collections.defaultdict(int)
    for key,values in data.items():
        key_list = key.split()
        for i in range(len(key_list) -1):
            pairwords[(key_list[i],key_list[i+1])] += values
    return pairwords


pairs = get_pair_with_frequency(train_data)
pairs

defaultdict(int,
            {('d', 'e'): 5,
             ('e', 'e'): 5,
             ('e', 'p'): 5,
             ('p', '</w>'): 5,
             ('l', 'e'): 7,
             ('e', 'a'): 7,
             ('a', 'r'): 7,
             ('r', 'n'): 7,
             ('n', 'i'): 7,
             ('i', 'n'): 14,
             ('n', 'g'): 17,
             ('g', '</w>'): 14,
             ('n', 'a'): 6,
             ('a', 't'): 6,
             ('t', 'u'): 6,
             ('u', 'r'): 6,
             ('r', 'a'): 6,
             ('a', 'l'): 6,
             ('l', '</w>'): 6,
             ('l', 'a'): 3,
             ('a', 'n'): 3,
             ('g', 'u'): 3,
             ('u', 'a'): 3,
             ('a', 'g'): 3,
             ('g', 'e'): 3,
             ('e', '</w>'): 3,
             ('p', 'r'): 7,
             ('r', 'o'): 7,
             ('o', 'c'): 7,
             ('c', 'e'): 7,
             ('e', 's'): 7,
             ('s', 's'): 7,
             ('s', 'i'): 7})

In [20]:
best_pair = max(pairs,key=pairs.get)
best_pair

('n', 'g')

In [21]:
def merge_data_with_pair(best_pair,data):
    result = {}
    compiles = re.compile(" ".join(best_pair))
    for word in data:
        new_word = re.sub(compiles,best_pair[0]+best_pair[1],word)
        result[new_word] = data[word]
    return result
train_data = merge_data_with_pair(best_pair, train_data)
print("语料库: ", train_data)

语料库:  {'d e e p </w>': 5, 'l e a r n i ng </w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3, 'p r o c e s s i ng </w>': 7}


In [22]:
def build_vocab(train_data,num_epoch):
    subwords = get_subwords(train_data)
    bpe_vocab = set(sorted(subwords,key=subwords.get))
    print(bpe_vocab,len(bpe_vocab))
    i = 1
    for _ in range(num_epoch):
        pairs = get_pair_with_frequency(train_data)
        if not pairs:
            break
        best_pair = max(pairs,key=pairs.get)
        if pairs[best_pair] == 1:
            break
        train_data = merge_data_with_pair(best_pair=best_pair,data=train_data)

        subwords = get_subwords(train_data)
        bpe_vocab = set(sorted(subwords,key=subwords.get))
        print("Iter - {}, 最高频子词对: {}".format(i, best_pair))
        print("训练数据: ", train_data)
        print("词表: {}, {}\n".format(len(bpe_vocab), bpe_vocab))
        i += 1

    return bpe_vocab

num_merges = 14

train_data = {'d e e p </w>': 5, 'l e a r n i n g </w>': 7, 'n a t u r a l </w>': 6, 'l a n g u a g e </w>': 3,'p r o c e s s i n g </w>':7}

bpe_vocab = build_vocab(train_data, num_merges)
print("词表: ", bpe_vocab)

{'a', 's', 't', 'c', 'l', 'd', 'i', 'u', 'n', 'o', 'r', 'p', 'g', '</w>', 'e'} 15
Iter - 1, 最高频子词对: ('n', 'g')
训练数据:  {'d e e p </w>': 5, 'l e a r n i ng </w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3, 'p r o c e s s i ng </w>': 7}
词表: 16, {'a', '</w>', 's', 't', 'c', 'l', 'd', 'ng', 'i', 'u', 'o', 'r', 'p', 'g', 'n', 'e'}

Iter - 2, 最高频子词对: ('i', 'ng')
训练数据:  {'d e e p </w>': 5, 'l e a r n ing </w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3, 'p r o c e s s ing </w>': 7}
词表: 16, {'a', '</w>', 's', 't', 'c', 'l', 'd', 'ng', 'ing', 'u', 'o', 'r', 'p', 'g', 'n', 'e'}

Iter - 3, 最高频子词对: ('ing', '</w>')
训练数据:  {'d e e p </w>': 5, 'l e a r n ing</w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3, 'p r o c e s s ing</w>': 7}
词表: 16, {'a', '</w>', 's', 't', 'c', 'l', 'e', 'd', 'ng', 'u', 'o', 'r', 'p', 'g', 'n', 'ing</w>'}

Iter - 4, 最高频子词对: ('l', 'e')
训练数据:  {'d e e p </w>': 5, 'le a r n ing</w>': 7, 'n a t u r a l </w>': 6, 'l a ng u a g e </w>': 3, 'p r o c 

In [23]:
import re 

def toknize_word(word,sorted_vocab,unknown_token='<UNK>'):
    if not word:
        return []
    if not sorted_vocab:
        return [unknown_token]*len(word)
    
    word_token = []
    for i in range(len(sorted_vocab)):
        token = sorted_vocab[i]
        token_reg = re.escape(token.replace('.', '[.]'))
        word = re.sub(token_reg,' '+token+' ',word)
        list_word = word.split()
        for other in list_word:
            if other == token:
                word_token+=[token]
            else:
                word_token += toknize_word(other, sorted_vocab[i+1:], unknown_token)
        return word_token

    return [unknown_token]*len(word)


In [24]:
def tokenize(text, bpe_vocab):
    sorted_vocab = sorted(bpe_vocab, key=lambda subword: len(subword), reverse=True)
    print("待编码语句: ", text)
    tokens = []
    for word in text.split():
        word = word + "</w>"
        word_tokens = toknize_word(word, sorted_vocab, unknown_token='<unk>')
        tokens.extend(word_tokens)
    
    return tokens

text = "natural language processing"
tokens = tokenize(text, bpe_vocab)
print("词表: ", bpe_vocab)
print("编码结果: ", tokens)

待编码语句:  natural language processing
词表:  {'a', 'process', '</w>', 't', 'l', 'learning</w>', 'd', 'ng', 'e', 'u', 'r', 'p', 'g', 'n', 'ing</w>'}
编码结果:  ['n', 'a', 't', 'u', 'r', 'a', 'l', '</w>', 'l', 'a', 'ng', 'u', 'a', 'g', 'e', '</w>', 'process', 'ing</w>']


In [25]:
def restore(tokens):
    text = []
    word = []
    for token in tokens:
        if token[-4:] == "</w>":
            if token!="</w>":
                word.append(token[:-4])
            text.append(''.join(word))
            word.clear()
        else:
            word.append(token)
        
    return text
text = "natural language processing"
tokens = tokenize(text, bpe_vocab)
print("词表: ", bpe_vocab)
print("编码结果: ", tokens)
text_r = restore(tokens=tokens)
print("还原结果: ", text_r)

待编码语句:  natural language processing
词表:  {'a', 'process', '</w>', 't', 'l', 'learning</w>', 'd', 'ng', 'e', 'u', 'r', 'p', 'g', 'n', 'ing</w>'}
编码结果:  ['n', 'a', 't', 'u', 'r', 'a', 'l', '</w>', 'l', 'a', 'ng', 'u', 'a', 'g', 'e', '</w>', 'process', 'ing</w>']
还原结果:  ['natural', 'language', 'processing']
